<a href="https://colab.research.google.com/github/myoungjinahn/Machine-Learning-project-team1/blob/main/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%EB%AA%A8%EB%8D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [64]:
# 데이터 로드
from google.colab import drive
drive.mount('/content/drive')

# 2. 파일 경로 설정 및 로드
# 파일이 '내 드라이브' 바로 아래에 있다면 아래 경로를 사용합니다.
file_path = '/content/drive/MyDrive/기온, 초미세먼지 데이터.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
df['comfortable'] = ((df['최고기온(℃)'] >= 15) & (df['최고기온(℃)'] <= 24) &
                     (df['미세먼지PM2.5'] <= 25)).astype(int)

In [66]:
# [수정] 실제 CSV 파일의 컬럼명과 일치시킵니다.
feature_cols = ['최고기온(℃)', '미세먼지PM2.5']
target_col = 'comfortable'

In [67]:
# 2) 전처리 및 분리
X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [68]:
# 3) 파이프라인 구성
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

In [69]:
# 4) 하이퍼파라미터 탐색
# 결과를 저장할 변수 초기화
best_k = 0
best_score = 0
k_scores = [] # 시각화를 위해 점수들을 저장할 리스트

# 1부터 10까지 k값을 바꿔가며 루프 수행, 수정 전 최적의 수치 11~20 사이
for k in range(1, 17):
    # 1. 해당 k값으로 모델 생성 (Pipeline 활용)
    temp_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])

    # 2. 모델 학습
    temp_pipe.fit(X_train, y_train)

    # 3. 테스트 데이터로 점수(정확도) 계산
    current_score = temp_pipe.score(X_test, y_test)
    k_scores.append(current_score)

    # 4. [핵심] if문을 사용해 최적의 점수와 k값 갱신
    if current_score > best_score:
        best_score = current_score
        best_k = k

    print(f"k가 {k}일 때의 정확도: {current_score:.4f}")

print("-" * 30)
print(f"최종 발견된 최적의 k: {best_k}")
print(f"그때의 최고 정확도: {best_score:.4f}")

k가 1일 때의 정확도: 0.9636
k가 2일 때의 정확도: 0.9455
k가 3일 때의 정확도: 0.9455
k가 4일 때의 정확도: 0.9273
k가 5일 때의 정확도: 0.9455
k가 6일 때의 정확도: 0.9455
k가 7일 때의 정확도: 0.9273
k가 8일 때의 정확도: 0.9636
k가 9일 때의 정확도: 0.9455
k가 10일 때의 정확도: 0.9455
k가 11일 때의 정확도: 0.9636
k가 12일 때의 정확도: 0.9636
k가 13일 때의 정확도: 0.9636
k가 14일 때의 정확도: 0.9636
k가 15일 때의 정확도: 0.9636
k가 16일 때의 정확도: 0.9636
------------------------------
최종 발견된 최적의 k: 1
그때의 최고 정확도: 0.9636


In [70]:
# 1. 훈련 데이터와 테스트 데이터 각각에 대한 예측값 생성
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# 2. 성능 지표 계산
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

# 3. 비교 결과 출력
print(" [ 성능 비교 결과 ]")
print(f"{'Metric':<12} | {'Train Set':<12} | {'Test Set':<12} | {'Difference'}")
print("-" * 55)
print(f"{'Accuracy':<12} | {train_acc:<12.4f} | {test_acc:<12.4f} | {abs(train_acc - test_acc):.4f}")
print(f"{'F1-Score':<12} | {train_f1:<12.4f} | {test_f1:<12.4f} | {abs(train_f1 - test_f1):.4f}")

# 4. 과적합 여부 판단 보조 메시지
if train_acc > test_acc + 0.05:
    print("\n 알림: 훈련 세트 성적이 테스트 세트보다 눈에 띄게 높습니다. 과적합(Overfitting) 가능성이 있습니다.")
else:
    print("\n 알림: 두 세트의 성적이 비슷합니다. 모델이 비교적 잘 일반화되었습니다.")

 [ 성능 비교 결과 ]
Metric       | Train Set    | Test Set     | Difference
-------------------------------------------------------
Accuracy     | 0.9633       | 0.9455       | 0.0178
F1-Score     | 0.9565       | 0.9333       | 0.0232

 알림: 두 세트의 성적이 비슷합니다. 모델이 비교적 잘 일반화되었습니다.


In [71]:
# 4) 최적의 모델 학습 및 최종 평가
best_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k))
])
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print('\n[최종 모델 평가 결과]')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred, zero_division=0))


[최종 모델 평가 결과]
Accuracy: 0.9636
Confusion Matrix:
 [[30  2]
 [ 0 23]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97        32
           1       0.92      1.00      0.96        23

    accuracy                           0.96        55
   macro avg       0.96      0.97      0.96        55
weighted avg       0.97      0.96      0.96        55

